# bartorch Library Internals: BartTensor, BartContext, and the Hot Path

This notebook goes beneath the high-level ops API and explains how bartorch works
internally.  Understanding the internals helps you:

- write code that stays on the **zero-copy hot path** when the C++ extension is built
- diagnose dispatch decisions (hot path vs. subprocess fallback)
- integrate bartorch with existing PyTorch pipelines
- prepare for the upcoming linear operator and iterative algorithm APIs (Phase 2/3)

**Topics covered:**
1. `BartTensor` — memory layout, dtype, and PyTorch compatibility
2. The dispatch graph — how bartorch chooses hot path vs. FIFO subprocess
3. `BartContext` — session-level CFL registry sharing
4. Full pipeline inside a context
5. Loading real CFL data from disk
6. Preview of the linear operator API (Phase 2)

In [ ]:
import torch
import numpy as np
import bartorch
import bartorch.ops as ops
from bartorch.core.tensor import BartTensor
from bartorch.core.context import bart_context
from bartorch.utils.cfl import readcfl, writecfl
print(f"bartorch version: {bartorch.__version__}")

## 1. BartTensor

`BartTensor` is a `torch.Tensor` subclass with two hard constraints that match
BART's internal C representation:

- **dtype** `torch.complex64` — corresponds directly to C's `complex float`
- **strides** Fortran (column-major) — BART stores all arrays with the fastest
  varying index first, the opposite of NumPy/C convention

These constraints make it safe to pass `tensor.data_ptr()` directly to BART's
`register_mem_cfl_non_managed()` without any copy or transpose.

In [ ]:
# BartTensor is a torch.Tensor subclass with complex64 dtype
# and Fortran (column-major) strides — matching BART's memory layout.

ph = ops.phantom([128, 128])
print(f"Type:   {type(ph)}")
print(f"Shape:  {ph.shape}")
print(f"Dtype:  {ph.dtype}")
print(f"Strides: {ph.stride()}")   # should be (1, 128) — Fortran order

# BartTensors are fully compatible with PyTorch
print(f"\nIs instance of torch.Tensor: {isinstance(ph, torch.Tensor)}")
print(f"Mean abs value: {ph.abs().mean().item():.4f}")

In [ ]:
# Convert to NumPy
arr_np = ph.numpy()
print(f"NumPy array shape: {arr_np.shape}, dtype: {arr_np.dtype}")

# Convert from NumPy  (promotes to BartTensor via dispatch)
arr_back = torch.from_numpy(arr_np)
print(f"Back to tensor: {arr_back.shape}")

# BartTensor can also hold GPU data (when CUDA is available)
if torch.cuda.is_available():
    ph_gpu = ph.cuda()
    print(f"GPU tensor device: {ph_gpu.device}")
else:
    print("CUDA not available — running CPU-only")

## 2. The Dispatch Graph

Every bartorch op routes through `bartorch.core.graph.dispatch()`, which selects
one of two execution paths:

```
dispatch(op_name, inputs, output_dims, **kwargs)
  │
  ├─ C++ extension available AND all inputs are BartTensors?
  │   → Path A (hot path): register data_ptr(), call bart_command(), zero copies
  │
  └─ C++ extension not built?
      → Path B (subprocess): write CFL pairs to /dev/shm, run BART binary,
        read output CFL, return BartTensor
```

Path B is fully correct and requires only a BART system binary.  Path A requires
the C++ extension to be compiled (`pip install -e .`).

On Path B, `/dev/shm` is used on Linux (a RAM-backed `tmpfs` — no physical disk
I/O).  On macOS / Windows the system temp directory is used instead.

In [ ]:
from bartorch.core.graph import dispatch

# dispatch() selects:
#   - Hot path:  C++ extension available + all inputs are BartTensors
#   - Fallback:  BART subprocess with CFL temp files in /dev/shm

# Check which path will be used:
try:
    import bartorch._bartorch_ext as _ext
    print("C++ extension available — hot path active")
except ImportError:
    print("C++ extension not built — using subprocess fallback")
    print("(Install with:  pip install -e .  or  BARTORCH_SKIP_EXT=1 pip install -e .)")

## 3. BartContext — Chained Operations

Without a `BartContext`, each op independently creates and tears down a CFL
registry mini-session: register inputs → run BART → unlink names.

With `bart_context()`, every op inside the `with` block shares the same
thread-local CFL registry session.  This eliminates redundant re-registration
overhead between consecutive ops and keeps all intermediate results as live
`*.mem` handles — so the next op can consume them without a round-trip through
Python memory.

On the hot path (C++ extension), this means consecutive ops execute entirely
inside C without ever returning to Python between calls.

In [ ]:
# Without a context, each op creates its own CFL registry mini-session.
# With bart_context(), all ops in the block share the same session,
# avoiding re-registration overhead for consecutive calls.

with bart_context() as ctx:
    # All three ops share the same in-memory CFL session
    phantom_img = ops.phantom([128, 128])
    kspace = ops.fft(phantom_img, flags=3)
    # The context cleans up the CFL registry on exit

print(f"kspace shape: {kspace.shape}")
print(f"kspace dtype: {kspace.dtype}")

## 4. Full Pipeline in a Context

The cell below runs a complete MRI acquisition-and-reconstruction pipeline
inside a single `BartContext`.  When the C++ extension is built, all three
BART operations execute without returning to Python between calls — the
intermediate k-space and sensitivity arrays are passed directly by their
`*.mem` handles.

In [ ]:
# Complete MRI acquisition + reconstruction pipeline
# (all ops stay in C when the extension is built)

with bart_context():
    # 1. Generate 8-coil phantom k-space
    kspace_mc = ops.phantom([128, 128], kspace=True, ncoils=8)

    # 2. Coil calibration
    sens = ops.ecalib(kspace_mc, calib_size=16)

    # 3. L1-Wavelet compressed sensing reconstruction
    reco = ops.pics(kspace_mc, sens, lambda_=0.005, iter_=30, wav=True)

print(f"Pipeline output shape: {reco.shape}")
print(f"Pipeline output dtype:  {reco.dtype}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ops.ifft(kspace_mc[..., 0], flags=3).squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Coil 1 (IFFT)')
axes[0].axis('off')

axes[1].imshow(sens[..., 0, 0].abs().numpy(), cmap='viridis')
axes[1].set_title('Coil 1 sensitivity (ESPIRiT)')
axes[1].axis('off')

axes[2].imshow(reco.squeeze().abs().numpy(), cmap='gray')
axes[2].set_title('PICS reconstruction')
axes[2].axis('off')

plt.suptitle('bartorch pipeline: phantom \u2192 ecalib \u2192 pics', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Working with Real CFL Data

`readcfl` returns a NumPy `complex64` array; `writecfl` accepts any array-like.
To feed real data into bartorch ops, convert to a tensor with `torch.as_tensor()`
(zero-copy when the array is already contiguous `complex64`) or let the bartorch
dispatch layer promote it automatically.

The dispatch layer will copy plain `torch.Tensor` inputs to a `BartTensor` when
needed — the copy is unavoidable if the layout does not match, but it happens at
most once per input per op call.

In [ ]:
# If you have BART's demo data (k-space saved as CFL files), you can load
# them directly and pass to bartorch ops.
#
# Example (replace paths with your actual data):
#
#   kspace = readcfl('/path/to/kspace')          # numpy array, complex64
#   kspace_bt = torch.as_tensor(kspace)          # promote to BartTensor
#   sens = ops.ecalib(kspace_bt, calib_size=24)
#   reco = ops.pics(kspace_bt, sens, lambda_=0.01, wav=True)

# For demonstration, we create synthetic data instead:
synth = np.random.randn(64, 64, 1, 4).astype(np.complex64)
synth += 1j * np.random.randn(64, 64, 1, 4).astype(np.float32)

import tempfile, os
with tempfile.TemporaryDirectory() as tmpdir:
    p = os.path.join(tmpdir, 'synth_kspace')
    writecfl(p, synth)
    loaded = readcfl(p)
    print(f"CFL round-trip: {synth.shape} \u2192 wrote \u2192 loaded {loaded.shape}")
    print(f"Max abs error:  {np.abs(synth - loaded).max():.2e}")

## 6. Linear Operators (Coming in Phase 2)

BART has a powerful internal linear operator framework (`struct linop_s*`) that
supports forward, adjoint, normal, and pseudo-inverse applications.  Operators
can be composed — chained, stacked, summed — without materialising intermediate
arrays.

Phase 2 will expose this as `BartLinop`, a Python object that holds an opaque
C++ handle to a `linop_s*`.  The planned API mirrors old bartpy's SWIG wrappers
but routes through the hot path:

In [ ]:
# The linop API is a stub; it will be wired to BART's linop_s framework
# in Phase 2. Here we show the planned interface:

from bartorch.ops.linops import BartLinop

# This will work once the C++ extension is built:
# fft_op = ops.fft_linop([256, 256], flags=3)
# sens_op = ops.diag([256, 256, 1, 8], diag_dims=[256, 256, 1, 8], flags=7)
# encoding = ops.chain(sens_op, fft_op)   # SENSE encoding operator
# 
# y = ops.forward(encoding, [256, 256, 1, 8], image)   # A x
# x = ops.adjoint(encoding, [256, 256], kspace)        # A^H y

print("BartLinop is a placeholder for BART's linop_s*:")
linop = BartLinop(src_dims=[256, 256], dst_dims=[256, 256, 1, 8])
print(f"  {linop}")
print("\nFull linop API will be available in Phase 2.")

## Summary

### Execution path decision table

| Condition | Path | Notes |
|-----------|------|-------|
| C++ ext built + all inputs `BartTensor` | **Hot path** (Path A) | Zero copies, bart_command() in-process |
| C++ ext built + mixed `torch.Tensor` / `ndarray` input | **Hot path after promotion** | One copy per non-BartTensor input |
| C++ ext not built | **Subprocess fallback** (Path B) | CFL pairs in `/dev/shm` (Linux) or temp dir |

### Roadmap

| Phase | Feature | Status |
|-------|---------|--------|
| 0 | Package skeleton, BartTensor, BartContext, FIFO fallback | ✅ Done |
| 1 | C++ extension: `_bartorch_ext.run()`, zero-copy hot path | 🔧 In progress |
| 2 | `BartLinop`, all linop constructors and composition | 📋 Planned |
| 3 | Iterative algorithms: conjgrad, IST, FISTA, IRGNM, CP | 📋 Planned |
| 4 | Auto-generated CLI wrappers for all 100+ BART tools | 📋 Planned |

See `01_high_level_tools.ipynb` for a practical walkthrough of the ops API.